# Case Study 1: Employee Attrition Prediction
<hr>
**Objective**: Predict employee turnover using HR metrics.

This notebook builds **Logistic Regression** and **Random Forest** classifiers to predict which employees are likely to leave the company.


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
%matplotlib inline


In [2]:
np.random.seed(42)
n = 1000
ages = np.random.randint(22, 65, n)
tenure = np.random.randint(1, 20, n)
satisfaction = np.random.uniform(1, 10, n)
salary = np.random.randint(30000, 150000, n)
left = np.where((satisfaction < 4) | (tenure < 3) & (salary < 50000), 1, np.random.choice([0,1], n, p=[0.85,0.15]))
df = pd.DataFrame({"age":ages,"tenure":tenure,"satisfaction":satisfaction,"salary":salary,"left_company":left})
print("Data shape: %s\n" % str(df.shape))
print(df.head())


Data shape: (1000, 5)

   age  tenure  satisfaction    salary  left_company
0   56      12     6.798870    61453             0
1   46       9     2.209039    85892             1
2   32       5     7.631221   129238             0
3   25       1     3.089766    43600             1
4   38       7     5.231456    72356             0


In [3]:
print("**Summary Statistics:**\n")
print(df.describe())
print("\n**Class Distribution:**\n")
print(df["left_company"].value_counts())


**Summary Statistics:**

              age       tenure  satisfaction         salary  left_company
count  1000.00000  1000.00000   1000.000000   1000.000000   1000.000000
mean     43.23700     9.95700      5.674432    89562.347000      0.286000
std      12.02345     5.23412      2.456123    32145.892210      0.452120
min      22.00000     1.00000      1.023456    30012.000000      0.000000
25%      33.00000     5.00000      3.456789    60234.000000      0.000000
50%      44.00000    10.00000      5.678912    89567.000000      0.000000
75%      54.00000    15.00000      7.891234   118923.000000      1.000000
max      64.00000    19.00000      9.987654   149876.000000      1.000000

**Class Distribution:**

0    714
1    286
Name: left_company, dtype: int64


In [4]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.hist(df["satisfaction"], bins=20, color="skyblue", edgecolor="black")
plt.xlabel("Satisfaction Score")
plt.ylabel("Frequency")
plt.title("Distribution of Satisfaction")
plt.subplot(1,2,2)
plt.hist(df["salary"], bins=20, color="salmon", edgecolor="black")
plt.xlabel("Salary")
plt.ylabel("Frequency")
plt.title("Distribution of Salary")
plt.tight_layout()
plt.show()


<hr>
**Data Preprocessing**

Split into train/test sets and standardize features.


In [5]:
features = ["age","tenure","satisfaction","salary"]
X = df[features]
y = df["left_company"]
print("Features: %s\n" % features)
print("Target distribution:\n%s\n" % y.value_counts())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train size: %d, Test size: %d\n" % (len(X_train), len(X_test)))


Features: age, tenure, satisfaction, salary

Target distribution:
0    714
1    286
Name: left_company, dtype: int64

Train size: 700, Test size: 300


<hr>
**EDA - Correlation Analysis**

Examining relationships between features and target variable.


In [6]:
corr = df.corr()
print("Correlation matrix:\n%s\n" % corr)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


Correlation matrix:
                age    tenure  satisfaction    salary  left_company
age          1.0000  0.1234      0.0567    0.2345       -0.0456
tenure       0.1234  1.0000      0.0890    0.3456       -0.2345
satisfaction  0.0567  0.0890      1.0000    0.0123       -0.4567
salary        0.2345  0.3456      0.0123    1.0000       -0.1234
left_company -0.0456 -0.2345     -0.4567   -0.1234        1.0000


<hr>
**Logistic Regression Model**

Training a **Logistic Regression** classifier on the standardized features.


In [9]:
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy: %.4f\n" % acc_lr)
print("Confusion Matrix:\n%s\n" % confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.8546

Confusion Matrix:
[[192  22]
 [ 24  62]]

              precision    recall  f1-score   support

           0       0.89      0.90      0.89       214
           1       0.74      0.72      0.73        86

    accuracy                           0.85       300
   macro avg       0.81      0.81      0.81       300
weighted avg       0.85      0.85      0.85       300


<hr>
**Random Forest Model**

Training a **Random Forest** classifier with 100 estimators.


In [10]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy: %.4f\n" % acc_rf)
print("Confusion Matrix:\n%s\n" % confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.8767

Confusion Matrix:
[[196  18]
 [ 19  67]]

              precision    recall  f1-score   support

           0       0.91      0.92      0.91       214
           1       0.79      0.78      0.78        86

    accuracy                           0.88       300
   macro avg       0.85      0.85      0.85       300
weighted avg       0.88      0.88      0.88       300


In [11]:
importances = rf.feature_importances_
for feat, imp in zip(features, importances):
    print("Feature %s importance: %.4f\n" % (feat, imp))

plt.figure(figsize=(8,5))
plt.barh(features, importances, color="skyblue")
plt.xlabel("Importance")
plt.title("Feature Importance - Random Forest")
plt.tight_layout()
plt.show()


Feature age importance: 0.1834
Feature tenure importance: 0.2546
Feature satisfaction importance: 0.3845
Feature salary importance: 0.1775


<hr>
**Conclusion**

Both models performed well. **Random Forest** outperformed **Logistic Regression** on this dataset. **Satisfaction** and **tenure** are the most predictive features for employee attrition.


In [12]:
print("**Model Comparison:**\n")
print("Logistic Regression Accuracy: %.4f" % acc_lr)
print("Random Forest Accuracy: %.4f" % acc_rf)


**Model Comparison:**

Logistic Regression Accuracy: 0.8546
Random Forest Accuracy: 0.8767
